# 🚀 TV5 / TOÀN NHÓM — BENCHMARK & CRAWLING MS1 TRÊN COLAB CPU-ONLY

**Dự án:** arXiv Data Science Pipeline (Intro to Data Science — HCMUS) — **Nhóm 6**  
**Yêu cầu đề tài:** *"The testbed for the final benchmarking is a Google Colab instance running on CPU-only mode."*  
**Cơ chế lưu trữ an toàn:** Tự động kết nối **Google Drive** để bảo toàn 100% dữ liệu và checkpoint (`state/stages`, `state/discovery`, `corpus/`), giúp khôi phục (`--resume`) mượt mà nếu phiên Colab bị ngắt.

---
### 📋 BẢNG PHÂN CHIA DẢI ID NHÓM 6 (`configs/ms1_group6_ranges.json`)
| Thành viên | Vai trò | Dải ID được giao | Số lượng | Thư mục lưu Google Drive |
| :--- | :--- | :--- | :---: | :--- |
| **TV1** | Leader & Discovery | `2508.12872:2508.17871` | 5.000 bài | `MyDrive/arxiv_ms1_data/tv1/` |
| **TV2** | Source Downloader | `2506.15788:2506.20787` | 5.000 bài | `MyDrive/arxiv_ms1_data/tv2/` |
| **TV3** | File Extractor | `2411.08799:2411.13798` | 5.000 bài | `MyDrive/arxiv_ms1_data/tv3/` |
| **TV4** | Metadata & S2 Refs | `2410.18025:2410.23024` | 5.000 bài | `MyDrive/arxiv_ms1_data/tv4/` |
| **TV5** | QA & Report | `2410.03025:2410.08024` | 5.000 bài | `MyDrive/arxiv_ms1_data/tv5/` |

> ⚠️ **HƯỚNG DẪN CHẠY ĐỘC LẬP:**  
> Mỗi dải 5.000 bài tốn khoảng 10-11 tiếng. **5 thành viên mở 5 tài khoản Colab độc lập để chạy song song dải của mình**. Cả nhóm sẽ hoàn thành toàn bộ 25.000 bài chỉ trong 1 đêm!

## 1. Gắn kết Google Drive (Bắt buộc để lưu Checkpoint bền vững)
> Dữ liệu tải về và checkpoint sẽ được lưu trực tiếp trên Google Drive. Khi Colab bị timeout sau vài tiếng, bạn chỉ cần mở lại notebook và chạy tiếp, hệ thống sẽ **tiếp tục ngay bài đang dở mà không bị mất dữ liệu**.

In [ ]:
from google.colab import drive
from pathlib import Path
import os, sys, shutil

# Mount Google Drive
drive.mount('/content/drive')

DRIVE_BASE = Path('/content/drive/MyDrive/arxiv_ms1_data')
DRIVE_BASE.mkdir(parents=True, exist_ok=True)
print('✅ Google Drive đã kết nối thành công!')
print(f'📁 Thư mục gốc lưu trữ: {DRIVE_BASE}')

## 2. Kiểm tra Môi trường phần cứng (Bắt buộc CPU-only)
> Đề tài yêu cầu benchmark ở chế độ CPU-only. Cell này sẽ kiểm tra xem máy có lỡ bật GPU hay không.

In [ ]:
import subprocess

assert 'google.colab' in sys.modules or __import__('importlib').util.find_spec('google.colab'), 'Notebook dành riêng cho Google Colab'

if shutil.which('nvidia-smi'):
    gpu = subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True)
    if gpu.returncode == 0 and 'GPU ' in gpu.stdout:
        raise AssertionError('⚠️ CẢNH BÁO: Máy đang bật GPU! Hãy vào menu Runtime -> Change runtime type -> Chọn CPU rồi chạy lại.')

print('✅ CHUẨN XÁC: Môi trường đang ở chế độ CPU-only (không có GPU accelerator).')
!lscpu | grep -E 'Model name|Socket|Thread|NUMA|CPU\(s\)'

## 3. Cài source đã kiểm chứng

Cố định commit `0f8d83b591c89cf085f344f5fb59af671ab43b81`, kiểm tra source sạch và chạy tests. Không fallback âm thầm sang nhánh khác; cell benchmark chỉ chạy khi setup thành công.

In [ ]:
from pathlib import Path
import os, sys, subprocess, json, datetime, shutil

SOURCE_READY = False
SOURCE_URL = 'https://github.com/LeoTkiet/arxiv-data-science.git'
SOURCE_REF = '4ea44c42810f3bb1564080c19d1ed889f3498ee0'
SOURCE = Path('/content/arxiv-data-science')
if not SOURCE.exists():
    subprocess.run(['git', 'clone', SOURCE_URL, str(SOURCE)], check=True)
if subprocess.check_output(['git', '-C', str(SOURCE), 'status', '--porcelain'], text=True).strip():
    raise RuntimeError('Source có thay đổi cục bộ; kiểm tra trước khi đổi commit.')
subprocess.run(['git', '-C', str(SOURCE), 'fetch', 'origin', SOURCE_REF], check=True)
subprocess.run(['git', '-C', str(SOURCE), 'checkout', '--detach', 'FETCH_HEAD'], check=True)
SOURCE_COMMIT = subprocess.check_output(['git', '-C', str(SOURCE), 'rev-parse', 'HEAD'], text=True).strip()
if SOURCE_COMMIT != SOURCE_REF:
    raise RuntimeError('Source commit không khớp phiên bản đã kiểm chứng.')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(SOURCE) + '[reporting,dev]'], check=True)
subprocess.run([sys.executable, '-m', 'pytest', 'tests/crawling', 'tests/test_validation.py',
                'tests/test_ms1_reporting.py', '-q'], check=True, cwd=str(SOURCE))
sys.path.insert(0, str(SOURCE / 'src'))
SOURCE_READY = True
print('Source đã kiểm chứng:', SOURCE_COMMIT)


## 4. Cấu hình Semantic Scholar API Key (Tùy chọn)
> Đặt `S2_API_KEY` trong **Colab Secrets** (biểu tượng chìa khóa ở thanh công cụ bên trái). Nếu không có key, hệ thống vẫn chạy bình thường với pool công khai.

In [ ]:
try:
    from google.colab import userdata
    key = userdata.get('S2_API_KEY')
    if key:
        os.environ['S2_API_KEY'] = key
        print('✅ Đã nạp S2_API_KEY từ Colab Secrets.')
    else:
        print('ℹ️ Chưa đặt S2_API_KEY trong Colab Secrets (dùng rate limit công khai).')
except Exception:
    print('ℹ️ Bỏ qua Colab Secrets.')

## 5. Chọn phạm vi

TV1–TV5 dùng một dải theo thứ tự config; đối chiếu tên/MSSV trong Excel trước khi chạy. `AUTO_RESUME` giữ corpus/checkpoint. Mỗi lần chạy benchmark tạo thư mục kết quả mới. `GROUP_OUTPUT=None` mặc định; chỉ điền đường dẫn khi đã tổng hợp corpus/checkpoints cả nhóm vào một output chung.

In [ ]:
# @title Chọn phạm vi chạy
MEMBER_ROLE = 'TV2' # @param ['TEST_SMOKE_1_BAI', 'TV1', 'TV2', 'TV3', 'TV4', 'TV5', 'ALL_GROUP_25K']
SAVE_TO_DRIVE = True # @param {type:'boolean'}
AUTO_RESUME = True # @param {type:'boolean'}
FAST_BATCH_PREFETCH = True # @param {type:'boolean'}
RESUME_START_ID = 'AUTO' # @param ['AUTO', 'START'] {allow-input: true}
RUN_BENCHMARK = False # @param {type:'boolean'}
# Chỉ đặt khi đã tổng hợp corpus/checkpoints của cả nhóm vào một output chung.
GROUP_OUTPUT = None

import datetime, json
assignment = json.loads((SOURCE / 'configs/ms1_group6_ranges.json').read_text(encoding='utf-8'))
RANGES_MAP = {
    'TEST_SMOKE_1_BAI': {'range': '2508.12872:2508.12872', 'folder': 'smoke_test'},
    **{f'TV{i + 1}': {'range': item['range'], 'folder': f'tv{i + 1}'}
       for i, item in enumerate(assignment['ranges'])},
    'ALL_GROUP_25K': {'folder': 'all_group6'},
}
cfg = RANGES_MAP[MEMBER_ROLE]
selected_ranges = assignment['ranges'] if MEMBER_ROLE == 'ALL_GROUP_25K' else [{'range': cfg['range']}]
from arxiv_pipeline.main import expand_range
paper_count = len({identifier for item in selected_ranges for identifier in expand_range(item['range'])})
scope_assignment = {'ranges': selected_ranges, 'assigned_input_count': paper_count,
                    'scope': MEMBER_ROLE, 'note': 'ID lookup inputs, not a count of existing publications'}

base_root = Path('/content/drive/MyDrive/arxiv_ms1_data') if SAVE_TO_DRIVE else Path('/content/ms1-data')
OUTPUT = base_root / cfg['folder'] / 'data'
RESULTS_ROOT = base_root / cfg['folder'] / 'artifacts'
RESULTS = None  # Được tạo tên mới mỗi lần chạy cell benchmark.
OUTPUT.mkdir(parents=True, exist_ok=True)

# Smart Fast Resume: Căn cứ DUY NHẤT vào file validation đã hoàn tất (không tính folder rỗng)
effective_ranges = list(selected_ranges)
if AUTO_RESUME and MEMBER_ROLE not in {'ALL_GROUP_25K', 'TEST_SMOKE_1_BAI'} and RESUME_START_ID != 'START':
    full_ids = expand_range(cfg['range'])
    month_prefix = full_ids[0].split('.')[0] + '.'
    val_dir = OUTPUT / 'validation'
    done_ids = []
    if val_dir.is_dir():
        for p in val_dir.glob(f'{month_prefix}*.json'):
            if p.is_file() and p.stat().st_size > 0 and p.stem in full_ids:
                done_ids.append(p.stem)
    if done_ids:
        max_done = max(done_ids)
        max_idx = full_ids.index(max_done)
        target_idx = max(0, max_idx - 5)
        new_start = full_ids[target_idx] if RESUME_START_ID == 'AUTO' else RESUME_START_ID
        if new_start in full_ids and full_ids.index(new_start) > 0:
            new_range = f"{new_start}:{full_ids[-1]}"
            effective_ranges = [{'range': new_range}]
            print(f"🔄 [Smart Fast Resume]: Đã thẩm định hoàn tất đến {max_done}. Tự động lùi 5 bài để tiếp tục từ {new_range} (bỏ qua {target_idx} bài cũ đã thẩm định).")

cli_args = [arg for item in effective_ranges for arg in ['--range', item['range']]]
if AUTO_RESUME:
    cli_args.append('--resume')
print({'member': MEMBER_ROLE, 'assigned_inputs': paper_count, 'output': str(OUTPUT),
       'results_parent': str(RESULTS_ROOT), 'resume': AUTO_RESUME,
       'free_disk_bytes': shutil.disk_usage(OUTPUT).free})
print('Đối chiếu tên/MSSV với Excel để xác nhận thứ tự TV1–TV5 trước khi chạy.')


## 6. Khởi chạy Pipeline & Benchmark
> Chạy thông qua module `benchmark.py` của TV5: Vừa crawl vừa liên tục lấy mẫu RAM, Disk, Wall-time chuẩn xác.  
> **Lưu ý:** Nếu phiên Colab bị ngắt sau vài tiếng, bạn chỉ cần chạy lại từ Cell 1 đến Cell 6 này, hệ thống sẽ tự động resume từ điểm dừng.

In [ ]:
assert RUN_BENCHMARK, 'Đổi RUN_BENCHMARK=True khi đã kiểm tra cấu hình.'
assert globals().get('SOURCE_READY'), 'Chạy thành công cell cài đặt và kiểm thử source trước.'
if not AUTO_RESUME and OUTPUT.exists() and any(OUTPUT.iterdir()):
    raise ValueError('Fresh run cần output rỗng; chọn output mới hoặc bật resume.')
import uuid, time
from types import SimpleNamespace
stamp = datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ')
RESULTS = RESULTS_ROOT / f'run_{stamp}_{uuid.uuid4().hex}'
if RESULTS.exists():
    raise FileExistsError('Thư mục kết quả đã tồn tại; giữ nguyên dữ liệu và chạy lại cell.')
env = dict(os.environ)
env['PYTHONPATH'] = os.pathsep.join(filter(None, [str(SOURCE / 'src'), env.get('PYTHONPATH')]))

if globals().get('FAST_BATCH_PREFETCH'):
    print('⚡ [Batch Pre-fetch] Bắt đầu gom lô Metadata arXiv và S2 References...')
    batch_args = [arg for item in effective_ranges for arg in ['--range', item['range']]]
    try:
        subprocess.run([sys.executable, '-m', 'arxiv_pipeline.crawling.discovery_batch',
                        '--output', str(OUTPUT), *batch_args], cwd=str(SOURCE), env=env)
    except Exception as exc:
        print('⚠️ Batch discovery notice:', exc)
    try:
        subprocess.run([sys.executable, '-m', 'arxiv_pipeline.crawling.s2.batch',
                        '--output', str(OUTPUT), *batch_args], cwd=str(SOURCE), env=env)
    except Exception as exc:
        print('⚠️ Batch S2 notice:', exc)

command = [sys.executable, '-m', 'arxiv_pipeline.reporting.milestone1.benchmark',
           '--output', str(OUTPUT), '--results', str(RESULTS), '--', *cli_args]
print('Command:', command)

if hasattr(subprocess, 'Popen'):
    start_time = time.time()
    proc = subprocess.Popen(command, cwd=str(SOURCE), env=env)
    timings_path = RESULTS / 'timings.jsonl'
    target_count = len({identifier for item in effective_ranges for identifier in expand_range(item['range'])})
    while proc.poll() is None:
        time.sleep(1.5)
        if timings_path.is_file():
            try:
                lines = timings_path.read_text(encoding='utf-8', errors='replace').splitlines()
                papers = [l for l in lines if '"kind": "paper"' in l or '"kind":"paper"' in l]
                finished = len(papers)
                elapsed = time.time() - start_time
                pct = (finished / target_count * 100) if target_count else 0
                speed = (elapsed / finished) if finished > 0 else 0
                eta_sec = (target_count - finished) * speed if target_count >= finished else 0
                eta_str = f"{int(eta_sec // 3600)}h {int((eta_sec % 3600) // 60)}m"
                last_line = lines[-1] if lines else ''
                last_id = ''
                if '"paper_id":' in last_line:
                    last_id = last_line.split('"paper_id":')[1].split(',')[0].strip(' "')
                print(f"\r🚀 Tiến độ: {finished}/{target_count} ({pct:.1f}%) | Bài: {last_id or 'đang xử lý'} | Tốc độ: {speed:.1f}s/bài | ETA: {eta_str}", end='', flush=True)
            except Exception:
                pass
    proc.wait()
    print()
    completed = SimpleNamespace(returncode=proc.returncode)
else:
    completed = subprocess.run(command, cwd=str(SOURCE), env=env)

print('Pipeline exit:', completed.returncode, 'Results:', RESULTS)
if not (RESULTS / 'benchmark.json').is_file():
    raise RuntimeError('Benchmark chưa tạo kết quả; xem lỗi cell trước khi audit/đóng gói.')
if completed.returncode:
    for name in ['pipeline.log', 'statistics_error.json']:
        path = RESULTS / name
        if path.exists():
            print(name, path.read_text(encoding='utf-8', errors='replace')[-2000:])


## 7. Tái kiểm snapshot và audit

Kiểm UUID/hash trước và sau revalidate rồi mới xuất lại statistics có cùng run ID/label. Audit cá nhân dùng đúng dải đã chọn; smoke dùng 1 ID. ALL_GROUP_25K audit toàn assignment trên output chung. Với lượt cá nhân, audit nhóm chỉ chạy nếu có GROUP_OUTPUT; không cộng số trong các file summary để giả làm kiểm tra toàn bộ corpus. Chỉ audit sau khi các tiến trình ghi dữ liệu đã dừng.

In [ ]:
assert RESULTS is not None and (RESULTS / 'benchmark.json').is_file(), 'Cần có lượt benchmark trước.'
from arxiv_pipeline.reporting.milestone1.benchmark import current_snapshot
from arxiv_pipeline.common.utils import atomic_json

def print_rates(rates):
    for name, rate in rates.items():
        value = rate['value']
        percent = 'không áp dụng/chưa có mẫu số' if value is None else f'{value * 100:.2f}%'
        print(f"{name}: {percent} ({rate['numerator']}/{rate['denominator']})")

report_path = RESULTS / 'benchmark.json'
report = json.loads(report_path.read_text(encoding='utf-8'))
if report.get('statistics_status') == 'current_run':
    before = current_snapshot(OUTPUT, report['run_id'])
    if before != report['snapshot']:
        raise ValueError('Snapshot không khớp benchmark; không xuất lại statistics.')
    subprocess.run([sys.executable, '-m', 'arxiv_pipeline.reporting.milestone1.statistics',
                    '--output', str(OUTPUT), '--save', str(RESULTS / 'statistics-revalidated.json'),
                    '--revalidate'], check=True, cwd=str(SOURCE))
    checked = json.loads((RESULTS / 'statistics-revalidated.json').read_text(encoding='utf-8'))
    if current_snapshot(OUTPUT, report['run_id']) != before:
        raise ValueError('Snapshot thay đổi khi tái kiểm; không xuất lại statistics.')
    checked.update(label=report['label'], run_id=report['run_id'])
    atomic_json(RESULTS / 'statistics.json', checked)
    print_rates(checked['rates'])
else:
    print('Không có snapshot hoàn tất của lượt hiện tại; giữ bằng chứng, không xuất statistics cũ.')
report['environment']['is_colab'] = True
report['environment']['cpu_only_confirmation'] = 'CPU selected manually; NVIDIA GPU probe passed'
atomic_json(report_path, report)

def run_assignment_audit(output, definition, filename, scope):
    import uuid
    attempt = RESULTS / ('audit-' + uuid.uuid4().hex)
    definition_path = attempt / 'inputs.json'
    atomic_json(definition_path, definition)
    saved = attempt / filename
    process = subprocess.run([sys.executable, '-m', 'arxiv_pipeline.reporting.milestone1.audit',
                              '--output', str(output), '--assignment', str(definition_path),
                              '--save', str(saved)], cwd=str(SOURCE))
    if process.returncode not in (0, 1) or not saved.is_file():
        raise RuntimeError(f'Audit {scope} không tạo được báo cáo mới.')
    data = json.loads(saved.read_text(encoding='utf-8'))
    unresolved = len(data['unresolved_discovery_ids'])
    print({'scope': scope, 'assigned': data['assigned_input_count'],
           'resolved': data['discovered_assigned_ids'] - unresolved,
           'unseen': len(data['unseen_ids']), 'unresolved': unresolved,
           'validation_pass': data['validation_pass_papers'],
           'validation_fail': data['validation_fail_papers'], 'complete': data['audit_complete']})
    atomic_json(RESULTS / filename, data)
    return data

audit_data = run_assignment_audit(OUTPUT, scope_assignment, 'assignment-audit.json', MEMBER_ROLE)
if MEMBER_ROLE == 'ALL_GROUP_25K':
    print('Audit trên đã dùng toàn bộ assignment nhóm và output chung.')
elif GROUP_OUTPUT is not None:
    group_output = Path(GROUP_OUTPUT)
    if not group_output.is_dir():
        raise ValueError('GROUP_OUTPUT phải là corpus/checkpoint chung đã tổng hợp.')
    group_audit_data = run_assignment_audit(group_output, assignment, 'group-assignment-audit.json', 'GROUP')
else:
    print('Chưa audit toàn nhóm: cần GROUP_OUTPUT trỏ đến corpus/checkpoint chung đã tổng hợp.')


## 8. Đóng gói Artifacts và Tải về
> Nén toàn bộ tệp kết quả (`benchmark.json`, `statistics.json`, `samples.jsonl`, `timings.jsonl`, `assignment-audit.json`) thành tệp zip. Tệp này được lưu trên Google Drive VÀ tự động tải về máy qua trình duyệt để bạn gửi cho TV5 đưa vào Jupyter Book.

In [ ]:
from google.colab import files

archive_base = f'/content/TV_{cfg["folder"]}_MS1_RESULTS_{RESULTS.name}'
archive_path = shutil.make_archive(archive_base, 'zip', RESULTS)

# Copy 1 bản lưu trên Google Drive cho an toàn
if SAVE_TO_DRIVE:
    shutil.copy(archive_path, DRIVE_BASE / f'TV_{cfg["folder"]}_MS1_RESULTS_{RESULTS.name}.zip')
    print(f'💾 Đã lưu 1 bản zip trên Google Drive tại: {DRIVE_BASE}')

print(f'🎉 TỆP KẾT QUẢ SẴN SÀNG: {archive_path}')
print('Đang kích hoạt tải về máy...')
try:
    files.download(archive_path)
except Exception as e:
    print('ℹ️ Tải qua trình duyệt bị chặn, bạn có thể lấy trực tiếp file trên Google Drive:', e)

---
### 💡 MẸO GIỮ PHIÊN GOOGLE COLAB KHÔNG BỊ DISCONNECT (KEEP-ALIVE)
Để tránh Colab tự ngắt khi chạy tác vụ dài 10-12 tiếng:
1. Nhấn phím `F12` (hoặc `Ctrl + Shift + I` trên Windows, `Cmd + Option + I` trên Mac) trên trình duyệt để mở **Developer Console**.
2. Chọn thẻ **Console**.
3. Dán đoạn mã sau vào và nhấn `Enter`:
```javascript
function ClickConnect(){
    console.log("Colab Keep-Alive: Dang giu ket noi...");
    document.querySelector("colab-connect-button")?.click() || document.querySelector("#connect")?.click();
}
setInterval(ClickConnect, 60000); // 1 phut click 1 lan
```